In [1]:
!pip install  -q torch torchvision opencv-python tqdm matplotlib

In [3]:
import os, json, math, random
from typing import Optional, Tuple

import numpy as np
import cv2
from tqdm import tqdm
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision

In [4]:
# Download FreiHAND
!wget https://lmb.informatik.uni-freiburg.de/data/freihand/FreiHAND_pub_v2.zip
!unzip -q FreiHAND_pub_v2.zip -d /content/FreiHAND

--2025-12-14 03:32:51--  https://lmb.informatik.uni-freiburg.de/data/freihand/FreiHAND_pub_v2.zip
Resolving lmb.informatik.uni-freiburg.de (lmb.informatik.uni-freiburg.de)... 132.230.167.23
Connecting to lmb.informatik.uni-freiburg.de (lmb.informatik.uni-freiburg.de)|132.230.167.23|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3881833583 (3.6G) [application/zip]
Saving to: ‘FreiHAND_pub_v2.zip’

FreiHAND_pub_v2.zip 100%[===================>]   3.62G  28.1MB/s    in 2m 19s  

2025-12-14 03:35:11 (26.6 MB/s) - ‘FreiHAND_pub_v2.zip’ saved [3881833583/3881833583]



In [6]:
# CONFIG
FREIHAND_ROOT = "/content/FreiHAND"

RGB_DIR = os.path.join(FREIHAND_ROOT, "training", "rgb")
XYZ_JSON = os.path.join(FREIHAND_ROOT, "training_xyz.json")
K_JSON = os.path.join(FREIHAND_ROOT, "training_K.json")

# image/heatmap size
IN_SIZE = 224
HM_SIZE = 56
SIGMA = 2.0

# training
BATCH_SIZE = 32
EPOCHS = 50
LR = 3e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 4

# split
VAL_RATIO = 0.1
SEED = 42

# crop margin
CROP_MARGIN = 0.25

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# Sanity checks
assert os.path.isdir(RGB_DIR), f"RGB_DIR not found: {RGB_DIR}"
assert os.path.isfile(XYZ_JSON), f"XYZ_JSON not found: {XYZ_JSON}"
assert os.path.isfile(K_JSON),   f"K_JSON not found: {K_JSON}"

Device: cpu


In [7]:
# 3D to 2D
def project_xyz_to_uv(xyz: np.ndarray, K: np.ndarray) -> np.ndarray:
    """
    Project 3D keypoints (J,3) -> 2D pixels (J,2) using camera intrinsics K (3,3).
    xyz is in camera coordinates; K gives fx, fy, cx, cy.
    """
    X = xyz[:, 0]
    Y = xyz[:, 1]
    Z = np.clip(xyz[:, 2], 1e-6, None)

    fx, fy = K[0, 0], K[1, 1]
    cx, cy = K[0, 2], K[1, 2]

    u = fx * (X / Z) + cx
    v = fy * (Y / Z) + cy
    return np.stack([u, v], axis=1)


def bbox_from_points(uv: np.ndarray, w: int, h: int, margin: float = 0.25) -> Tuple[int,int,int,int]:
    """
    Create a square-ish bbox around 2D points with margin.
    Returns (x0,y0,x1,y1) in inclusive-exclusive format.
    """
    x_min, y_min = uv.min(axis=0)
    x_max, y_max = uv.max(axis=0)

    bw = x_max - x_min
    bh = y_max - y_min
    side = max(bw, bh) * (1.0 + margin)

    cx = (x_min + x_max) * 0.5
    cy = (y_min + y_max) * 0.5

    x0 = int(round(cx - side * 0.5))
    y0 = int(round(cy - side * 0.5))
    x1 = int(round(cx + side * 0.5))
    y1 = int(round(cy + side * 0.5))

    # clamp to image
    x0 = max(0, x0); y0 = max(0, y0)
    x1 = min(w, x1); y1 = min(h, y1)

    # ensure non-empty
    if x1 <= x0 + 1: x1 = min(w, x0 + 2)
    if y1 <= y0 + 1: y1 = min(h, y0 + 2)

    return x0, y0, x1, y1


def crop_and_resize(img: np.ndarray, uv: np.ndarray, bbox: Tuple[int,int,int,int], out_size: int = 224):
    """
    Crop bbox from img, resize to out_size x out_size.
    Transform uv into resized crop coordinates.
    Returns: crop_resized, uv_crop_resized, valid_mask
    """
    x0, y0, x1, y1 = bbox
    crop = img[y0:y1, x0:x1]
    ch, cw = crop.shape[:2]

    crop_resized = cv2.resize(crop, (out_size, out_size), interpolation=cv2.INTER_LINEAR)

    uv_crop = uv.copy()
    uv_crop[:, 0] = (uv[:, 0] - x0) * (out_size / max(cw, 1))
    uv_crop[:, 1] = (uv[:, 1] - y0) * (out_size / max(ch, 1))

    valid = (uv_crop[:, 0] >= 0) & (uv_crop[:, 0] < out_size) & (uv_crop[:, 1] >= 0) & (uv_crop[:, 1] < out_size)
    return crop_resized, uv_crop, valid.astype(np.float32)
